# 예제 III · Response-Based Knowledge Distillation

사전 학습된 **ResNet-18 Teacher의 출력 분포**를 경량 ResNet-18 Student에 전달합니다. 비교할 두 Student는 같은 초기 가중치로 출발합니다. 한 쪽은 정답만, 다른 쪽은 정답과 Teacher의 Soft Target을 사용합니다.

[공개 Teacher 체크포인트](https://huggingface.co/edadaltocg/resnet18_cifar10)를 사용하며, 학습·평가는 CIFAR-10에서 수행합니다.

## 1. 데이터와 실행 환경

CIFAR-10 학습 데이터에서 **1만 장**을 고정해 두 Student를 각각 **5 epoch** 학습합니다. 평가는 같은 CIFAR-10 테스트 이미지 **1만 장**에서 진행합니다.

**코드 셀 1 · 실행 장치와 KD 학습 조건**

시드 50, 배치 크기 64, AdamW 학습률 0.002, 학습 Temperature 2, KD 가중치 0.25를 설정합니다.

In [ ]:
# 두 Student의 초기 가중치, 학습 데이터, 배치 순서에 같은 시드를 적용한다.
import copy
import os
import time
from pathlib import Path

# 새 CUDA 런타임에서 동일한 설정으로 다시 실행하기 위한 옵션이다.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
from torch import nn
from torch.nn import functional as F
from torchvision import datasets

SEED = 50
BATCH_SIZE = 64
TRAIN_SAMPLES = 10_000
EPOCHS = 5
LEARNING_RATE = 0.002
KD_TEMPERATURE = 2.0
KD_WEIGHT = 0.25

DATA_ROOT = Path(os.environ.get("AX_LAB_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_LAB_CACHE", "./student_lab_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

**코드 셀 2 · Teacher 기준으로 CIFAR-10 준비**

공개 Teacher에 맞춘 RGB 정규화를 적용합니다. 고정된 학습 이미지 1만 장과 전체 테스트 이미지 1만 장을 준비합니다.

이미 내려받은 CIFAR-10은 런타임의 캐시에서 읽습니다. 첫 실행에서는 공유 Google Drive에서 받고, 실패하면 CIFAR-10의 공식 주소를 사용합니다.

In [ ]:
from http.client import HTTPException
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlopen

from torchvision.datasets.utils import check_integrity


class DriveCIFAR10(datasets.CIFAR10):
    # 압축 해제된 데이터나 검증된 압축 파일을 먼저 재사용한다.
    def download(self):
        if self._check_integrity():
            return

        root = Path(self.root)
        archive = root / self.filename
        if not check_integrity(archive, self.tgz_md5):
            drive_url = (
                "https://drive.usercontent.google.com/download"
                "?id=1Sx382Bt_XcAUj9_igoCgWoNQ6c5ooSa6&export=download&confirm=t"
            )
            try:
                root.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(dir=root) as temp_dir:
                    staged = Path(temp_dir) / self.filename
                    print("Google Drive에서 CIFAR-10을 내려받습니다.")
                    with urlopen(drive_url, timeout=30) as response:
                        if response.headers.get_content_type() == "text/html":
                            raise ValueError("Drive가 파일 대신 안내 페이지를 반환했습니다.")
                        with staged.open("wb") as output:
                            copyfileobj(response, output, length=1024 * 1024)
                    if not check_integrity(staged, self.tgz_md5):
                        raise ValueError("CIFAR-10 원본 체크섬과 일치하지 않습니다.")
                    staged.replace(archive)
            except (OSError, HTTPException, ValueError):
                print("Drive에서 CIFAR-10을 받지 못해 공식 다운로드 주소로 다시 시도합니다.")

        # torchvision이 원본 체크섬을 확인하고 필요한 경우 압축을 해제한다.
        super().download()



# Teacher와 두 Student에 같은 RGB 정규화를 적용한다.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)
rgb_mean = torch.tensor(CIFAR_MEAN, device=DEVICE)[None, :, None, None]
rgb_std = torch.tensor(CIFAR_STD, device=DEVICE)[None, :, None, None]

train_data = DriveCIFAR10(DATA_ROOT, train=True, download=True)
test_data = DriveCIFAR10(DATA_ROOT, train=False, download=True)

def normalized_subset(dataset, indices):
    images = torch.from_numpy(dataset.data[indices.numpy()]).to(DEVICE)
    images = images.permute(0, 3, 1, 2).contiguous().float()
    images.div_(255).sub_(rgb_mean).div_(rgb_std)
    labels = torch.tensor(dataset.targets, device=DEVICE)[indices.to(DEVICE)]
    return images, labels

# 학습용 분할에서 1만 장을 고정하고, 평가에는 공식 테스트 분할 전체를 사용한다.
train_generator = torch.Generator().manual_seed(SEED)
train_indices = torch.randperm(len(train_data), generator=train_generator)[:TRAIN_SAMPLES]
train_images, train_labels = normalized_subset(train_data, train_indices)
test_images, test_labels = normalized_subset(test_data, torch.arange(len(test_data)))

# Temperature 그래프는 기존 실습에서 사용한 테스트 이미지 한 장으로 그린다.
preview_generator = torch.Generator().manual_seed(SEED + 1)
preview_index = torch.randperm(len(test_data), generator=preview_generator)[0].item()
preview_image = test_images[preview_index:preview_index + 1]
print(f"학습 {len(train_labels):,}장 / 테스트 {len(test_labels):,}장 / {EPOCHS} epoch")

## 2. Teacher 가중치 로드 및 Student 구조

사전 학습된 ResNet-18을 Teacher로 사용합니다. Teacher는 평가 모드로 고정하고, 학습할 때는 Student만 업데이트합니다.

**코드 셀 3 · 경량 Student 구조 정의**

예제 II의 Depthwise Separable ResNet-18을 두 학습 조건의 공통 Student 구조로 정의합니다.

In [ ]:
# 기존 BasicBlock의 3×3 연산을 Depthwise와 Pointwise로 나누어 경량 모델을 만든다.
def build_light_resnet18():
    __all__ = ['ResNet', 'resnet18']

    def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
        """3x3 convolution with padding"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                         padding=dilation, groups=in_planes, bias=False, dilation=dilation)

    def conv1x1(in_planes, out_planes, stride=1):
        """1x1 convolution"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

    # 주 경로에서 특징을 계산하고 Residual 입력을 더한다. 크기가 바뀌는 블록에서는 Residual도 변환한다.
    class BasicBlock(nn.Module):
        expansion = 1

        def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                     base_width=64, dilation=1, norm_layer=None):
            super(BasicBlock, self).__init__()

            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            if groups != 1 or base_width != 64:
                raise ValueError(
                    'BasicBlock only supports groups=1 and base_width=64')
            if dilation > 1:
                raise NotImplementedError(
                    "Dilation > 1 not supported in BasicBlock")
            # Both self.conv1 and self.downsample layers downsample the input when stride != 1
            # 3×3 Depthwise에서 채널별 공간 특징을 계산하고, 1×1 Pointwise에서 채널을 조합한다.
            self.dwconv33_1 = conv3x3(inplanes, planes, stride)
            self.conv11_1 = conv1x1(planes, planes)
            self.bn1 = norm_layer(planes)
            self.relu = nn.ReLU(inplace=True)

            self.dwconv33_2 = conv3x3(planes, planes)
            self.conv11_2 = conv1x1(planes, planes)
            self.bn2 = norm_layer(planes)
            self.downsample = downsample
            self.stride = stride

        def forward(self, x):
            identity = x

            out = self.dwconv33_1(x)
            out = self.conv11_1(out)
            out = self.bn1(out)
            out = self.relu(out)

            out = self.dwconv33_2(out)
            out = self.conv11_2(out)
            out = self.bn2(out)

            if self.downsample is not None:
                identity = self.downsample(x)

            out += identity
            out = self.relu(out)

            return out

    # 입력 stem과 Residual stage를 연결하고, Global Average Pooling 뒤에서 클래스별
    # Logit을 계산한다.
    class ResNet(nn.Module):

        def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                     groups=1, width_per_group=64, replace_stride_with_dilation=None,
                     norm_layer=None):
            super(ResNet, self).__init__()
            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            self._norm_layer = norm_layer

            self.inplanes = 64
            self.dilation = 1
            if replace_stride_with_dilation is None:
                # each element in the tuple indicates if we should replace
                # the 2x2 stride with a dilated convolution instead
                replace_stride_with_dilation = [False, False, False]
            if len(replace_stride_with_dilation) != 3:
                raise ValueError("replace_stride_with_dilation should be None "
                                 "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
            self.groups = groups
            self.base_width = width_per_group
            self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                                   bias=False)

            self.bn1 = norm_layer(self.inplanes)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            # 네 stage는 64·128·256·512채널을 사용한다. 첫 블록의 stride로 해상도를 조정한다.
            self.layer1 = self._make_layer(block, 64, layers[0])
            self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                           dilate=replace_stride_with_dilation[0])
            self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                           dilate=replace_stride_with_dilation[1])
            self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                           dilate=replace_stride_with_dilation[2])
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            self.fc = nn.Linear(512 * block.expansion, num_classes)

            # Convolution과 정규화 계층의 학습 초기값을 설정한다.
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

            # Zero-initialize the last BN in each residual branch,
            # so that the residual branch starts with zeros, and each residual block behaves like an identity.
            # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
            if zero_init_residual:
                for m in self.modules():
                    if isinstance(m, Bottleneck):
                        nn.init.constant_(m.bn3.weight, 0)
                    elif isinstance(m, BasicBlock):
                        nn.init.constant_(m.bn2.weight, 0)

        # 첫 블록에서 필요한 Residual 변환을 적용하고, 뒤의 블록은 같은 출력 크기로 이어 붙인다.
        def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
            norm_layer = self._norm_layer
            downsample = None
            previous_dilation = self.dilation
            if dilate:
                self.dilation *= stride
                stride = 1
            if stride != 1 or self.inplanes != planes * block.expansion:
                downsample = nn.Sequential(
                    conv1x1(self.inplanes, planes * block.expansion, stride),
                    norm_layer(planes * block.expansion),
                )

            layers = []
            layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                                self.base_width, previous_dilation, norm_layer))
            self.inplanes = planes * block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, planes, groups=self.groups,
                                    base_width=self.base_width, dilation=self.dilation,
                                    norm_layer=norm_layer))

            return nn.Sequential(*layers)

        # 입력에서 stem, 네 stage, Pooling, 최종 Linear Layer로 이어지는 순전파다.
        def _forward_impl(self, x):
            # See note [TorchScript super()]
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)
            x = self.maxpool(x)

            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)

            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.fc(x)

            return x

        def forward(self, x):
            return self._forward_impl(x)

    # CIFAR-10에 맞게 첫 Conv2d를 3×3, stride 1로 설정하고 출력 클래스 수를 10으로 둔다.
    def resnet_18_cifar_lightweight():
        r"""
            Modify the resnet 18 network in order to run on cifar-10 dataset

            To enhance the accuracy, the (kernel_size, stride, padding) of conv1 is modified to (3, 1, 1)
            referenced by <https://github.com/akamaster/pytorch_resnet_cifar10>
        """
        model = ResNet(block=BasicBlock, layers=[2, 2, 2, 2], num_classes=10)

        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1,
                                bias=False)

        return model

    return resnet_18_cifar_lightweight()

**코드 셀 4 · 정확도와 모델 크기 측정 함수**

테스트 분류 정확도, Conv2d와 Linear의 MACs, 파라미터 수를 집계하는 함수를 정의합니다.

In [ ]:
# 전체 테스트 이미지에서 예측 클래스와 정답이 일치한 비율을 구한다.
@torch.inference_mode()
def evaluate(model, images, labels):
    model.eval()
    correct = [
        (model(batch_images).argmax(1) == batch_labels).sum()
        for batch_images, batch_labels in zip(images.split(BATCH_SIZE * 2), labels.split(BATCH_SIZE * 2))
    ]
    return 100 * torch.stack(correct).sum().item() / len(labels)


# Conv2d와 Linear에 Hook을 붙여 이미지 한 장의 Dense MACs를 구하고 전체 파라미터 수를 센다.
def macs_and_parameters(model, input_shape=(3, 32, 32)):
    # Conv2d와 Linear의 multiply-accumulate만 집계.
    operations = 0
    def count(module, args, output):
        nonlocal operations
        if isinstance(module, nn.Conv2d):
            kernel = module.kernel_size[0] * module.kernel_size[1]
            operations += output.numel() * (module.in_channels // module.groups) * kernel
        else:
            operations += output.numel() * module.in_features
    # 실행 중 필요한 계층에만 Hook을 연결하고 블록이 끝나면 제거한다.
    hooks = [m.register_forward_hook(count) for m in model.modules()
             if isinstance(m, (nn.Conv2d, nn.Linear))]
    previous_mode = model.training
    try:
        model.eval()
        with torch.inference_mode():
            model(torch.zeros(1, *input_shape, device=DEVICE))
    finally:
        for hook in hooks:
            hook.remove()
        model.train(previous_mode)
    return operations, sum(p.numel() for p in model.parameters())

**코드 셀 5 · Teacher 가중치 로드와 Student 초기 상태 준비**

고정된 Teacher의 전체 테스트 정확도를 확인합니다. 셀 7에서 사용할 Student 초기 가중치를 시드 50으로 준비합니다.

In [ ]:
from torchvision.models import resnet18

# CIFAR-10으로 사전 학습한 공개 Teacher의 체크포인트를 불러온다.
TEACHER_URL = ("https://huggingface.co/edadaltocg/resnet18_cifar10/resolve/"
               "ccacbae365101b9bfa45e1f17f4acac30c2601b2/pytorch_model.bin")
teacher = resnet18(weights=None, num_classes=10)
teacher.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
teacher.maxpool = nn.Identity()
state_dict = torch.hub.load_state_dict_from_url(
    TEACHER_URL, model_dir=str(ASSET_ROOT), file_name="teacher.bin",
    map_location="cpu", weights_only=True
)
teacher.load_state_dict(state_dict, strict=True)
# Offline KD에서는 Teacher를 평가 모드에 두고 가중치를 고정한다.
teacher = teacher.to(DEVICE).eval().requires_grad_(False)
torch.manual_seed(SEED)
student_initial = build_light_resnet18()
print(f"Teacher 정확도 ({len(test_labels):,}장): {evaluate(teacher, test_images, test_labels):.2f}%")

## 3. Soft Target과 KD Loss

**Temperature T**는 Softmax 계산 전에 각 클래스의 Logit을 나누는 값입니다. 같은 이미지에서 **T=1**과 학습에 쓰는 **T=2**의 상위 세 클래스 확률을 비교합니다.

**코드 셀 6 · Temperature와 Response KD Loss**

Teacher의 Soft Target을 만들고, 같은 Temperature를 적용한 Student 로그 확률과의 KL Divergence에 T²를 곱해 KD Loss를 계산합니다.

In [ ]:
# 두 모델에 같은 Temperature를 적용한다. Teacher 확률은 셀 7에서 한 번 계산한다.
def soft_targets(logits, temperature):
    return F.softmax(logits / temperature, dim=1)

def response_kd(student_logits, teacher_probs, temperature):
    student_log_prob = F.log_softmax(student_logits / temperature, dim=1)
    return F.kl_div(student_log_prob, teacher_probs, reduction="batchmean") * temperature**2


with torch.inference_mode():
    teacher_logits = teacher(preview_image)
    # 같은 영상의 Logit을 사용해 T만 바꿨을 때 상위 확률이 어떻게 달라지는지 비교한다.
    for temperature in (1.0, KD_TEMPERATURE):
        probs, indices = (teacher_logits / temperature).softmax(1).topk(3)
        print(f"T={temperature:g}:", [(test_data.classes[i], round(p, 3))
                                    for p, i in zip(probs[0].tolist(), indices[0].tolist())])

## 4. 정답만 학습한 Student와 KD Student 비교

- **정답만 학습** · 실제 클래스에 대한 Cross-Entropy를 사용합니다.
- **정답 + KD** · 실제 클래스에 대한 Cross-Entropy에 Teacher의 Soft Target을 따르는 KD Loss를 추가합니다.

**코드 셀 7 · 동일한 초기 상태에서 Student 두 모델 학습**

같은 초기 가중치와 배치 순서로 두 모델을 5 epoch 학습합니다. 매 epoch에서 각각의 평균 CE와 테스트 정확도를 확인합니다. KD Loss는 KD Student에 대해 따로 출력하며, 마지막 줄에 두 정확도의 차이를 %p로 표시합니다.

학습에는 셀 1의 `KD_TEMPERATURE`를 사용합니다. 같은 설정에서 다시 비교할 때는 셀 7만 재실행하면 됩니다.

In [ ]:
# 고정된 Teacher로 학습 이미지의 Soft Target을 한 번 계산해 5 epoch 동안 재사용한다.
start = time.perf_counter()
with torch.no_grad():
    teacher_logits = torch.cat([teacher(images) for images in train_images.split(BATCH_SIZE)])
    teacher_probs = soft_targets(teacher_logits, KD_TEMPERATURE)

# 셀 7을 다시 실행해도 두 Student는 같은 초기 가중치에서 학습을 시작한다.
student_ce = copy.deepcopy(student_initial).to(DEVICE)
student_kd = copy.deepcopy(student_initial).to(DEVICE)
optim_ce = torch.optim.AdamW(student_ce.parameters(), lr=LEARNING_RATE)
optim_kd = torch.optim.AdamW(student_kd.parameters(), lr=LEARNING_RATE)

def train_epoch(model, optimizer, batches, targets=None):
    model.train()
    ce_total = torch.zeros((), device=DEVICE)
    kd_total = torch.zeros((), device=DEVICE)

    for indices in batches:
        images = train_images[indices].contiguous()
        labels = train_labels[indices]

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        ce_loss = F.cross_entropy(logits, labels)
        loss = ce_loss

        if targets is not None:
            kd_loss = response_kd(logits, targets[indices], KD_TEMPERATURE)
            loss = (1 - KD_WEIGHT) * ce_loss + KD_WEIGHT * kd_loss
            kd_total += kd_loss.detach() * len(indices)

        loss.backward()
        optimizer.step()
        ce_total += ce_loss.detach() * len(indices)

    return ce_total.item() / len(train_labels), kd_total.item() / len(train_labels)

# 매 epoch 같은 배치 순서를 공유하고, 두 모델의 정확도를 동일한 테스트 이미지에서 측정한다.
batch_generator = torch.Generator(device=DEVICE).manual_seed(SEED)
print(f"Seed={SEED} | 학습 T={KD_TEMPERATURE:g} | KD 가중치={KD_WEIGHT:g} | AdamW lr={LEARNING_RATE:g}")
for epoch in range(1, EPOCHS + 1):
    order = torch.randperm(len(train_labels), generator=batch_generator, device=DEVICE)
    batches = order.split(BATCH_SIZE)
    ce_avg, _ = train_epoch(student_ce, optim_ce, batches)
    kd_ce_avg, kd_avg = train_epoch(student_kd, optim_kd, batches, teacher_probs)

    ce_acc = evaluate(student_ce, test_images, test_labels)
    kd_acc = evaluate(student_kd, test_images, test_labels)
    print(
        f"Epoch {epoch}/{EPOCHS} | 정답만 CE {ce_avg:.3f}, 정확도 {ce_acc:.2f}% | "
        f"정답 + KD CE {kd_ce_avg:.3f}, KD Loss {kd_avg:.3f}, 정확도 {kd_acc:.2f}%"
    )

print(f"최종 정확도 | 정답만 {ce_acc:.2f}% → 정답 + KD {kd_acc:.2f}% ({kd_acc - ce_acc:+.2f}%p)")
print(f"실행 시간 {time.perf_counter() - start:.1f}초")